# Lekcija 16: Valići i Gaborovi filtri

Fourierova transformacija iz Lekcije 15 govori *koje* su frekvencije prisutne u signalu, ali ne i *gdje* &mdash; sinusna bazna funkcija proteže se preko cijele slike, pa se lokalizirana značajka, poput ruba ili dijela teksture, raspršuje preko cijelog spektra. **Valići** (engl. *wavelets*) i **Gaborovi filtri** dva su različita rješenja tog problema: oba se temelje na malim, prostorno lokaliziranim oscilacijama umjesto beskonačnih sinusoida, dajući zajedničku informaciju o *položaju* i *frekvenciji*.


In [ ]:
import numpy as np
import cv2
import pywt
import matplotlib.pyplot as plt

## Haarov valić: najjednostavniji mogući valić

Haarova valićna transformacija rastavlja signal na **aproksimaciju**, odnosno lokalne prosjeke, i **detalje**, odnosno lokalne razlike, izračunate na parovima uzoraka koji se ne preklapaju:

$$a_k = \frac{x_{2k} + x_{2k+1}}{\sqrt{2}}, \qquad d_k = \frac{x_{2k} - x_{2k+1}}{\sqrt{2}}$$

To bi trebalo izgledati poznato: u osnovi je riječ o istoj ideji „zamućivanje i zadržavanje ostatka” kao kod Laplacianove piramide iz Lekcije 13, samo izračunatoj po parovima umjesto Gaussovom jezgrom i bez preklapanja susjedstava.


In [ ]:
signal = np.array([1, 3, 5, 11, 7, 9, 2, 4], dtype=np.float64)

approx = (signal[0::2] + signal[1::2]) / np.sqrt(2)
detail = (signal[0::2] - signal[1::2]) / np.sqrt(2)

cA_ref, cD_ref = pywt.dwt(signal, 'haar')
print('manual matches pywt.dwt?', np.allclose(approx, cA_ref) and np.allclose(detail, cD_ref))
print('approximation (cA):', np.round(approx, 2))
print('detail (cD):       ', np.round(detail, 2))

## Od Haara do Daubechies: glatkiji valići

Haarov valić diskontinuiran je, odnosno ima nagli skok, zbog čega slabo lokalizira frekvencije &mdash; njegov vlastiti frekvencijski sadržaj široko je rasprostranjen, suprotno onome što želimo. **Daubechiesini valići** (<a href="../references.html#daubechies-1988">Daubechies, 1988</a>) koriste dulje, glatkije filtre s više *iščezavajućih momenata*, prihvaćajući širi prostorni nosač radi znatno boljeg frekvencijskog ponašanja. Valić `db2`, koji se ponekad naziva „D4” zbog 4 koeficijenta filtra, sljedeći je korak prema većoj glatkoći nakon Haarova valića.


In [ ]:
db2 = pywt.Wavelet('db2')
print('db2 low-pass (scaling) filter :', np.round(db2.dec_lo, 4))
print('db2 high-pass (wavelet) filter:', np.round(db2.dec_hi, 4))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].stem(db2.dec_lo)
axes[0].set_title('db2 scaling filter (low-pass)')
axes[1].stem(db2.dec_hi)
axes[1].set_title('db2 wavelet filter (high-pass)')
plt.tight_layout()
plt.show()

Prethodna Haarova transformacija samo je konvolucija i smanjenje uzorkovanja faktorom 2 s filtrima s 2 koeficijenta $[\tfrac{1}{\sqrt2}, \tfrac{1}{\sqrt2}]$ i $[\tfrac{1}{\sqrt2}, -\tfrac{1}{\sqrt2}]$. Daubechiesini valići slijede potpuno isti postupak s duljim filtrima. Dekompoziciju valića `db2` implementiramo od početka kao periodičnu konvoluciju i provjeravamo je usporedbom s bibliotekom `pywt`.


In [ ]:
def dwt_periodic(x, filt):
    """One level of a periodic discrete wavelet transform with an arbitrary filter."""
    n, L = len(x), len(filt)
    k = np.arange(n // 2)[:, None]
    i = np.arange(L)[None, :]
    idx = (2 * k + 2 - i) % n
    return (filt[None, :] * x[idx]).sum(axis=1)

longer_signal = np.array([1, 3, 5, 11, 7, 9, 2, 4, 6, 8], dtype=np.float64)

cA_mine = dwt_periodic(longer_signal, np.array(db2.dec_lo))
cD_mine = dwt_periodic(longer_signal, np.array(db2.dec_hi))
cA_ref, cD_ref = pywt.dwt(longer_signal, db2, mode='periodization')

print('matches pywt (periodization mode)?', np.allclose(cA_mine, cA_ref) and np.allclose(cD_mine, cD_ref))

reconstructed = pywt.idwt(cA_mine, cD_mine, db2, mode='periodization')
print('perfect reconstruction?', np.allclose(reconstructed, longer_signal))

## Dvodimenzionalna valićna dekompozicija slike

Poput separabilnog Gaussova filtra iz Lekcije 10, dvodimenzionalna valićna transformacija primjenjuje se kroz dva jednodimenzionalna prolaza: prvo po retcima, zatim po stupcima. Jedna razina dekompozicije rastavlja sliku na **četiri podpojasa**:

- **LL**: niskopropusno filtriranje u oba smjera &mdash; grublja kopija slike s prepolovljenom razlučivošću po osi, poput jedne razine Gaussove piramide
- **LH**: niskopropusno po retcima, visokopropusno po stupcima &mdash; reagira na vodoravne rubove
- **HL**: visokopropusno po retcima, niskopropusno po stupcima &mdash; reagira na okomite rubove
- **HH**: visokopropusno u oba smjera &mdash; reagira na dijagonalne detalje i kutove


In [ ]:
img = np.zeros((200, 200), dtype=np.float64)
cv2.rectangle(img, (40, 40), (160, 160), 200, -1)
cv2.circle(img, (100, 100), 40, 100, -1)

LL, (LH, HL, HH) = pywt.dwt2(img, 'db2', mode='periodization')

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, im, title in zip(axes, [img, LL, LH, HL, HH],
                          ['Original', 'LL (approx.)', 'LH (horiz. edges)', 'HL (vert. edges)', 'HH (diagonal)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Kao i Laplacianova piramida, ova dekompozicija može se točno invertirati: funkcija `pywt.idwt2` rekonstruira izvornik iz četiriju podpojasa bez gubitaka.


In [ ]:
reconstructed_img = pywt.idwt2((LL, (LH, HL, HH)), 'db2', mode='periodization')
print('max reconstruction error:', np.abs(reconstructed_img - img).max())

## Gaborovi filtri: lokalizirane oscilacije prilagođene orijentaciji

**Gaborov filtar** sinusoidalni je prugasti uzorak pomnožen Gaussovom ovojnicom &mdash; prostorno lokaliziran val prilagođen određenoj frekvenciji *i* orijentaciji. Za razliku od Daubechiesinih valića, namijenjenih kompaktnoj, ortogonalnoj i invertibilnoj dekompoziciji na više razlučivosti, Gaborovi filtri više se koriste za *izdvajanje značajki*: detekciju usmjerenih tekstura i rubova na odabranoj skali.

Gaborovi filtri imaju i zanimljivu biološku poveznicu. Snimajući aktivnost neurona u vidnom korteksu mačke, Hubel i Wiesel (<a href="../references.html#hubel-wiesel-1959">Hubel i Wiesel, 1959</a>) pronašli su „jednostavne stanice” koje selektivno reagiraju na prugu ili rub određene orijentacije i položaja, a gotovo uopće ne reagiraju na druge orijentacije &mdash; temeljno otkriće u neuroznanosti vida. Kasnije je pokazano (<a href="../references.html#marcelja-1980">Marcelja, 1980</a>; <a href="../references.html#daugman-1985">Daugman, 1985</a>) da je dvodimenzionalna Gaborova funkcija vrlo dobar matematički model receptivnih polja tih jednostavnih stanica. To je jedan od razloga zašto su Gaborovi filtri postali standardan, biološki motiviran alat za izdvajanje značajki osjetljivih na orijentaciju u računalnom vidu.


In [ ]:
orientations_deg = [0, 45, 90, 135]
wavelengths = [4, 8, 12, 16]

fig, axes = plt.subplots(len(wavelengths), len(orientations_deg), figsize=(11, 11))
for i, lambd in enumerate(wavelengths):
    for j, t in enumerate(orientations_deg):
        kernel = cv2.getGaborKernel((25, 25), sigma=3, theta=np.radians(t), lambd=lambd, gamma=0.5, psi=0)
        ax = axes[i, j]
        ax.imshow(kernel, cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(fr'$\theta$ = {t} deg', fontsize=9)
        if j == 0:
            ax.set_ylabel(fr'$\lambda$ = {lambd}', fontsize=9)
plt.tight_layout()
plt.show()

### Kvantitativni prikaz selektivnosti prema orijentaciji

Crtamo četiri pruge u četirima različitim orijentacijama i filtriramo sliku Gaborovom jezgrom prilagođenom svakoj orijentaciji, a zatim mjerimo prosječan iznos odziva u blizini svake pruge. Filtar bi trebao najjače reagirati na prugu koja odgovara njegovoj orijentaciji, a slabo na ostale.


In [ ]:
def draw_bar(image, center, angle_deg, length=60, thickness=6, value=255):
    angle = np.radians(angle_deg)
    dx, dy = length / 2 * np.cos(angle), length / 2 * np.sin(angle)
    p1 = (int(center[0] - dx), int(center[1] - dy))
    p2 = (int(center[0] + dx), int(center[1] + dy))
    cv2.line(image, p1, p2, value, thickness)

bar_orientations = [0, 45, 90, 135]
centers = [(50, 50), (150, 50), (50, 150), (150, 150)]

bars_img = np.zeros((200, 200), dtype=np.float64)
for c, ang in zip(centers, bar_orientations):
    draw_bar(bars_img, c, ang)

plt.imshow(bars_img, cmap='gray')
plt.title('Bars at 0, 45, 90, 135 degrees')
plt.axis('off')
plt.show()

In [ ]:
responses = np.zeros((4, 4))
for fi, bar_angle in enumerate(bar_orientations):
    # Note: cv2's theta parameter is the orientation of the sinusoidal stripes inside the
    # kernel, which runs *perpendicular* to the bar it responds to -- hence the +90 offset.
    kernel = cv2.getGaborKernel((25, 25), sigma=4, theta=np.radians(bar_angle + 90), lambd=10, gamma=0.5, psi=0)
    response = cv2.filter2D(bars_img, cv2.CV_64F, kernel)
    for ci, c in enumerate(centers):
        region = response[c[1] - 20:c[1] + 20, c[0] - 20:c[0] + 20]
        responses[fi, ci] = np.abs(region).mean()

print(f'{"filter tuned for":>18}', ' '.join(f'{a:>6}' for a in bar_orientations), '  <- bar orientation')
for fi, ang in enumerate(bar_orientations):
    print(f'{ang:>18}', ' '.join(f'{v:>6.0f}' for v in responses[fi]))

best_match = [bar_orientations[i] for i in responses.argmax(axis=1)]
print('\neach filter peaks at its own bar orientation?', best_match == bar_orientations)

Matrica odziva izrazito je dijagonalno dominantna: najveći odziv svakog filtra javlja se upravo na pruzi kojoj je prilagođen, što odgovara selektivnosti prema orijentaciji koju su Hubel i Wiesel uočili u biološkom sustavu.


### Zadaci

1. Isprobajte `db4` (`pywt.Wavelet('db4')`) umjesto `db2` za dvodimenzionalnu dekompoziciju slike. Kako se mijenja izgled podpojasa LH/HL/HL s obzirom na dulje, glatkije filtre valića `db4`?
2. Primijenite `pywt.dwt2` još jednom na podpojas `LL` prethodne dekompozicije kako biste dobili drugu, grublju razinu &mdash; to je **valićna piramida**, izravno analogna Gaussovim i Laplacianovim piramidama iz Lekcija 10 i 12.
3. Povećajte `lambd`, odnosno valnu duljinu sinusoide, u Gaborovoj jezgri uz fiksnu vrijednost `sigma`. Što se događa s brojem vidljivih pruga unutar Gaussove ovojnice i kako bi to moglo promijeniti skalu teksture stvarne slike na koju filtar reagira?
